# Predicción de Gravedad de Siniestros Viales — CABA
## Pipeline completo: Ingesta → Limpieza → Features → Modelo → Visualización

**Datasets utilizados (todos de data.buenosaires.gob.ar):**
1. Siniestros viales (Hechos): Información sobre Lesiones en siniestros viales ocurridos en la Ciudad. Los datos incluyen fecha y ubicación del hecho y tipo de transporte involucrado. Además se especifica el género y edad de las víctimas y el tipo de lesión sufrida.

2. Siniestros viales (Víctimas): Información sobre Homicidios en siniestros viales ocurridos en la Ciudad. Los datos incluyen fecha y ubicación del hecho y tipo de transporte involucrado. Además se especifica el género y edad de las víctimas y el tipo de lesión sufrida.

3. Flujo vehicular por radares AUSA (sensores autopista): Flujo vehicular captado por radares desagregado por hora. Incluye fecha, hora, nombre de la autopista y ubicación del radar.

4. Flujo vehicular — Anillo Digital (sensores ciudad): Listado con informacion de flujo vehicular detectado por los sensores de la Ciudad. Incluye latitud y longitud del sensor, fecha, hora y cantidad registrada.

5. Registro de precipitaciones: Registro de lluvias dividido por mes y año. Incluye milímetros de agua acumulados y días de precipitaciones por cada mes

6. Registro de temperatura: Registro de temperaturas máximas, medias y mínimas en la Ciudad dividido por mes y año

## 1. Imports y configuración global

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visualización
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid')

print("Librerías cargadas correctamente")


Librerías cargadas correctamente


## 2. Ingesta de datos

* Usamos gdown para descargar la carpeta `data/` desde un drive personal público en sólo lectura con los CSVs ya descargados del portal del gobierno de la ciudad

* Usaremos datos de 2021-2025


In [2]:
def cargar_csv(path, nombre, encoding='latin-1', sep=','):
    try:
        df = pd.read_csv(path, encoding=encoding, sep=sep, low_memory=False)
        print(f"✅ {nombre}: {df.shape[0]:,} filas × {df.shape[1]} columnas")
        return df
    except Exception as e:
        print(f"⚠️  {nombre}: error al cargar — {e}")
        return None

BASE = "https://raw.githubusercontent.com/valenmendez01/TPO-CienciaDeDatos/main/datasets/"

df_hechos       = cargar_csv(BASE + 'siniestros_viales_hechos.csv',         'Siniestros — Hechos',    sep=';')
df_victimas     = cargar_csv(BASE + 'siniestros_viales_victimas.csv',       'Siniestros — Víctimas',  sep=';')
df_flujo_anillo = cargar_csv(BASE + 'dataset_flujo_vehicular.csv',          'Flujo Anillo Digital')
df_ausa_2021    = cargar_csv(BASE + 'flujo-vehicular-por-radares-2021.csv', 'Flujo AUSA 2021',        sep=';')
df_ausa_2022    = cargar_csv(BASE + 'flujo-vehicular-por-radares-2022.csv', 'Flujo AUSA 2022',        sep=';')
df_ausa_2023    = cargar_csv(BASE + 'flujo-vehicular-por-radares-2023.csv', 'Flujo AUSA 2023',        sep=';')
df_ausa_2024    = cargar_csv(BASE + 'flujo-vehicular-por-radares-2024.csv', 'Flujo AUSA 2024',        sep=';')
df_ausa_2025    = cargar_csv(BASE + 'flujo-vehicular-por-radares-2025.csv', 'Flujo AUSA 2025',        sep=';')
df_lluvia       = cargar_csv(BASE + 'historico_precipitaciones.csv',        'Precipitaciones')
df_temp         = cargar_csv(BASE + 'historico_temperaturas.csv',           'Temperatura')


✅ Siniestros — Hechos: 54,064 filas × 21 columnas
✅ Siniestros — Víctimas: 1,048,575 filas × 9 columnas
✅ Flujo Anillo Digital: 189,814 filas × 6 columnas
✅ Flujo AUSA 2021: 261,245 filas × 12 columnas
✅ Flujo AUSA 2022: 108,997 filas × 9 columnas
✅ Flujo AUSA 2023: 237,911 filas × 9 columnas
✅ Flujo AUSA 2024: 237,911 filas × 9 columnas
✅ Flujo AUSA 2025: 607,073 filas × 9 columnas
✅ Precipitaciones: 423 filas × 4 columnas
✅ Temperatura: 423 filas × 5 columnas


## 3. Exploración inicial (EDA rápido)

In [3]:
def explorar(df, nombre):
    # Si el dataframe no se cargó correctamente, avisamos y salimos
    if df is None:
        print(f"⚠️  {nombre} no disponible")
        return

    print(f"\n{'='*55}")
    print(f"  📊 {nombre}")
    print(f"{'='*55}")

    # Filas y columnas del dataset
    print(f"  Forma (filas, columnas)    : {df.shape}")

    # Filas exactamente iguales en todas sus columnas
    print(f"  Duplicados                 : {df.duplicated().sum()}")

    # Tipo de dato de cada columna (int, float, object, datetime...)
    # Importante para saber si hay columnas numéricas leídas como texto
    print(f"\n  Columnas y tipos:")
    print(df.dtypes.to_string())

    # Cantidad de valores faltantes por columna
    # Solo muestra las columnas que tienen al menos un nulo
    print(f"\n  Nulos por columna:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")

    # Muestra las primeras 3 filas para ver el contenido real
    print(f"\n  Primeras filas:")
    display(df.head(3))

# Exploramos cada dataset cargado
#explorar(df_hechos,       'Siniestros — Hechos')
#explorar(df_victimas,     'Siniestros — Víctimas')
for año, df in {2021: df_ausa_2021, 2022: df_ausa_2022, 2023: df_ausa_2023,
                2024: df_ausa_2024, 2025: df_ausa_2025}.items():
    explorar(df, f'Flujo AUSA {año}')
#explorar(df_flujo_anillo, 'Flujo Anillo Digital')
#explorar(df_lluvia,       'Precipitaciones')
#explorar(df_temp,         'Temperatura')


  📊 Flujo AUSA 2021
  Forma (filas, columnas)    : (261245, 12)
  Duplicados                 : 0

  Columnas y tipos:
Mes                  object
 DÃ­a               float64
 AÃ±o de H_Fecha     object
H_Hora               object
Aut Nombre           object
Disp Nombre          object
Disp Ubicacion       object
Seccion Sentido      object
Disp Lat              int64
Disp Lng            float64
H_Cant_Veh          float64
Hora de H_Fecha     float64

  Nulos por columna:
 DÃ­a               22017
Disp Nombre           704
Disp Ubicacion      22663
Seccion Sentido     22663
Disp Lng           239228
H_Cant_Veh         261245
Hora de H_Fecha    261245

  Primeras filas:


,Mes,DÃ­a,AÃ±o de H_Fecha,H_Hora,Aut Nombre,Disp Nombre,Disp Ubicacion,Seccion Sentido,Disp Lat,Disp Lng,H_Cant_Veh,Hora de H_Fecha
0,1/1/2021,0.0,AU 4 Lugones,RD171 Esma,9.9,A,NaN,-58.459.669.999.999.900,943,NaN,NaN,NaN
1,1/1/2021,1.0,AU 4 Lugones,RD171 Esma,9.9,A,NaN,-58.459.669.999.999.900,3946,NaN,NaN,NaN
2,1/1/2021,2.0,AU 4 Lugones,RD171 Esma,9.9,A,NaN,-58.459.669.999.999.900,4662,NaN,NaN,NaN



  📊 Flujo AUSA 2022
  Forma (filas, columnas)    : (108997, 9)
  Duplicados                 : 2551

  Columnas y tipos:
DÃ­a, Mes, AÃ±o de H_Fecha     object
Hora de H_Fecha               float64
Aut Nombre                     object
Disp Nombre                    object
Disp Ubicacion                 object
Seccion Sentido                object
Disp Lat                       object
Disp Lng                       object
H_Cant_Veh                    float64

  Nulos por columna:
  → Sin nulos

  Primeras filas:


,"DÃ­a, Mes, AÃ±o de H_Fecha",Hora de H_Fecha,Aut Nombre,Disp Nombre,Disp Ubicacion,Seccion Sentido,Disp Lat,Disp Lng,H_Cant_Veh
0,2022-01-01,0.0,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",654.0
1,2022-01-01,1.0,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",2675.0
2,2022-01-01,2.0,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",3260.0



  📊 Flujo AUSA 2023
  Forma (filas, columnas)    : (237911, 9)
  Duplicados                 : 0

  Columnas y tipos:
ï»¿DÃ­a, Mes, AÃ±o de H_Fecha    object
Hora de H_Fecha                   int64
Aut Nombre                       object
Disp Nombre                      object
Disp Ubicacion                   object
Seccion Sentido                  object
Disp Lat                         object
Disp Lng                         object
H_Cant_Veh                        int64

  Nulos por columna:
  → Sin nulos

  Primeras filas:


,"ï»¿DÃ­a, Mes, AÃ±o de H_Fecha",Hora de H_Fecha,Aut Nombre,Disp Nombre,Disp Ubicacion,Seccion Sentido,Disp Lat,Disp Lng,H_Cant_Veh
0,1/1/2023,0,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",887
1,1/1/2023,1,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",3443
2,1/1/2023,2,AU 4 Lugones,RD165 Dorrego,"4,74",A,"-34,56336","-58,41529",4383



  📊 Flujo AUSA 2024
  Forma (filas, columnas)    : (237911, 9)
  Duplicados                 : 6705

  Columnas y tipos:
Día, Mes, Año de H_Fecha     object
Hora de H_Fecha             float64
Aut Nombre                   object
Disp Nombre                  object
Disp Ubicacion               object
Seccion Sentido              object
Disp Lat                     object
Disp Lng                     object
H_Cant_Veh                  float64

  Nulos por columna:
Día, Mes, Año de H_Fecha    6706
Hora de H_Fecha             6706
Aut Nombre                  6706
Disp Nombre                 6706
Disp Ubicacion              6706
Seccion Sentido             6706
Disp Lat                    6706
Disp Lng                    6706
H_Cant_Veh                  6706

  Primeras filas:


,"Día, Mes, Año de H_Fecha",Hora de H_Fecha,Aut Nombre,Disp Nombre,Disp Ubicacion,Seccion Sentido,Disp Lat,Disp Lng,H_Cant_Veh
0,1/1/2024,0.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",1344.0
1,1/1/2024,1.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",4798.0
2,1/1/2024,2.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",5522.0



  📊 Flujo AUSA 2025
  Forma (filas, columnas)    : (607073, 9)
  Duplicados                 : 10

  Columnas y tipos:
Día, Mes, Año de H_Fecha     object
Hora de H_Fecha             float64
Aut Nombre                   object
Disp Nombre                  object
Disp Ubicacion               object
Seccion Sentido              object
Disp Lat                     object
Disp Lng                     object
H_Cant_Veh                  float64

  Nulos por columna:
Día, Mes, Año de H_Fecha      11
Hora de H_Fecha               11
Aut Nombre                    11
Disp Nombre                   11
Disp Ubicacion                11
Seccion Sentido             1230
Disp Lat                      11
Disp Lng                      11
H_Cant_Veh                    11

  Primeras filas:


,"Día, Mes, Año de H_Fecha",Hora de H_Fecha,Aut Nombre,Disp Nombre,Disp Ubicacion,Seccion Sentido,Disp Lat,Disp Lng,H_Cant_Veh
0,1/1/2025,0.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",1317.0
1,1/1/2025,1.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",4863.0
2,1/1/2025,2.0,AU 4 Lugones,RD171 Esma,"9,9",A,"-34,53767","-58,45967",5808.0


## 4. Limpieza y Normalización de Datasets

### Limpieza df_hechos

* Sin duplicados

Conversiones tipo de dato:
* fecha_siniestro: object → datetime
* hora_siniestro: object ("13:40:00") → entero (hora del día 0-23)
* gravedad_siniestro: target ordinal → se encodea (LEVE=0, GRAVE=1, MORTAL=2)

Nulos:
* Valores SD/sd en columnas categóricas → unificados a "DESCONOCIDO"
* tipo_de_via_siniestro: 12.227 nulos → imputados con "DESCONOCIDO"
* hora_siniestro: 77 nulos (SD) → Nan
* rango_horario: 77 nulos (coinciden con SD en hora_siniestro) → NaN

Descartes:
* direccion_normalizada_siniestro: 12.900 nulos (23%) → se descarta
* geocodificacion_plana: coordenadas en formato WKT, redundante con longitud/latitud → se descarta
* participantes_siniestro: columna derivada de modo_desplazamiento_victima + contraparte → se descarta

Mantenemos:
* comuna_siniestro: 3.017 nulos → se mantiene para modelo, se excluye en visualización de mapa
* longitud/latitud: 2.378 nulos → ídem

In [4]:
def limpiar_hechos(df):
    df = df.copy()

    # Unificar SD/sd → DESCONOCIDO + normalizar texto en columnas object
    cols_sd = df.select_dtypes("object").columns
    df[cols_sd] = df[cols_sd].apply(lambda col: col.str.strip().str.upper().replace("SD", "DESCONOCIDO"))

    # Descartes
    df.drop(columns=["direccion_normalizada_siniestro", "geocodificacion_plana",
                      "participantes_siniestro"], errors="ignore", inplace=True)

    # hora_siniestro: "13:40:00" → entero (0-23), SD → NaN
    df["hora_siniestro"] = (
        pd.to_datetime(df["hora_siniestro"], format="%H:%M:%S", errors="coerce")
        .dt.hour
    )

    # rango_horario: recalcular los 77 nulos desde hora_siniestro
    df["rango_horario"] = df["rango_horario"].fillna(df["hora_siniestro"] // 6)

    # fecha_siniestro → datetime
    df["fecha_siniestro"] = pd.to_datetime(df["fecha_siniestro"], errors="coerce")

    # tipo_de_via: imputar nulos con DESCONOCIDO
    df["tipo_de_via_siniestro"] = df["tipo_de_via_siniestro"].fillna("DESCONOCIDO")

    # Target ordinal
    gravedad_map = {"LEVE": 0, "GRAVE": 1, "MORTAL": 2}
    df["gravedad_encoded"] = df["gravedad_siniestro"].map(gravedad_map)

    print(f"✅ Hechos limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    nulos_restantes = df.isnull().sum()
    print(nulos_restantes[nulos_restantes > 0].to_string())
    return df

### Limpieza df_victimas

* 986.499 filas completamente vacías → eliminadas (artefacto de exportación desde Excel)
* 7 duplicados → eliminados
* Resultado: ~42.643 filas válidas (2021–2024)

Conversiones tipo de dato:
* edad_victima: object → numérico
* fecha_siniestro: object → datetime
* anio_siniestro: float → entero

Correcciones:
* GRAVEdad_victima → renombrada a gravedad_victima
* Valores SD → unificados a DESCONOCIDO

Nulos:
* rol_victima: 210 nulos → imputados con DESCONOCIDO
* edad_victima: 13.012 nulos (30%) → se mantiene, se excluye en modelo si es necesario

Descartes:
* fecha_fallecimiento_victima: 99% nulos (solo aplica a casos MORTAL, información redundante con gravedad_victima) → se descarta

In [5]:
def limpiar_victimas(df):
    if df is None: return None
    df = df.copy()

    df.dropna(how='all', inplace=True)
    df.drop_duplicates(inplace=True)

    df.rename(columns={"GRAVEdad_victima": "gravedad_victima"}, inplace=True)

    cols_obj = df.select_dtypes("object").columns
    df[cols_obj] = df[cols_obj].apply(lambda col: col.str.strip().str.upper().replace("SD", "DESCONOCIDO"))

    df["edad_victima"] = pd.to_numeric(df["edad_victima"], errors="coerce")
    df["fecha_siniestro"] = pd.to_datetime(df["fecha_siniestro"], errors="coerce")

    # rol_victima: 210 nulos → DESCONOCIDO
    df["rol_victima"] = df["rol_victima"].fillna("DESCONOCIDO")

    # fecha_fallecimiento_victima: 99% nulos, solo útil en MORTAL → se descarta
    df.drop(columns=["fecha_fallecimiento_victima"], inplace=True)

    # anio_siniestro: float → int
    df["anio_siniestro"] = df["anio_siniestro"].astype("Int64")

    df = df[df["anio_siniestro"].between(2021, 2024)].reset_index(drop=True)

    print(f"✅ Víctimas limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")
    return df

df_victimas = limpiar_victimas(df_victimas)

✅ Víctimas limpio: 42,650 filas × 8 columnas
edad_victima    13012


### Limpieza df_flujo_ausa

* Se limpian por separado antes de unificar por diferencias estructurales entre años

2021:
* H_Cant_Veh (volumen vehicular) 100% nula → año descartado, no aporta dato útil
* Estructura de 12 columnas distinta al resto (9 columnas)

2022–2025 (estructura común):
* Primera columna con nombre roto (comas/BOM/encoding) → renombrada a "fecha"
* Encoding roto en nombres de columnas (latin-1) → corregido
* 2022: 2.551 duplicados → eliminados
* 2024: 6.706 filas completamente nulas + 6.705 duplicados → eliminados
* 2025: 10 duplicados + 11 filas nulas → eliminados

Conversiones tipo de dato:
* fecha: object → datetime
* hora: object → entero (0–23)
* volumen_vehicular: object → float
* lat/lng: object con coma decimal → float

Correcciones:
* Columnas renombradas a nombres consistentes y en minúscula
* sentido: nulos → DESCONOCIDO

Resultado: 2022–2025 unificados (~1.191.000 filas)

In [6]:
def _normalizar_fecha_ausa(serie):
    """Prueba YYYY-MM-DD, luego D/M/YYYY, luego serial de Excel."""
    resultado = pd.to_datetime(serie, format="%Y-%m-%d", errors="coerce")

    mask = resultado.isna()
    if mask.any():
        resultado[mask] = pd.to_datetime(serie[mask], format="%d/%m/%Y", errors="coerce")

    mask2 = resultado.isna()
    if mask2.any():
        resultado[mask2] = pd.to_datetime(serie[mask2], dayfirst=True, errors="coerce")

    # Seriales de Excel (números como '45474') → fecha real
    mask3 = resultado.isna()
    if mask3.any():
        seriales = pd.to_numeric(serie[mask3], errors="coerce")
        validos = seriales.notna()
        if validos.any():
            origen = pd.Timestamp("1899-12-30")  # origen de Excel
            resultado[mask3] = seriales.apply(
                lambda x: origen + pd.Timedelta(days=x) if pd.notna(x) else pd.NaT
            )

    return resultado

def _fix_coords(serie):
    """Reemplaza coma decimal por punto y convierte a float."""
    return pd.to_numeric(
        serie.astype(str).str.replace(",", ".").str.strip(),
        errors="coerce"
    )

def limpiar_ausa_2022_2025(df, año):
    """Para 2022–2025: misma estructura, distintos problemas menores."""
    df = df.copy()

    # Renombrar primera columna (nombre con comas/BOM/encoding roto) → "fecha"
    df.rename(columns={df.columns[0]: "fecha"}, inplace=True)

    # Normalizar nombres de columnas
    df.columns = (df.columns
                  .str.strip()
                  .str.encode("latin-1").map(lambda b: b.decode("utf-8", errors="replace"))
                  .str.lower()
                  .str.replace(" ", "_"))
    df.rename(columns={
        "hora_de_h_fecha": "hora",
        "aut_nombre":      "autopista",
        "disp_nombre":     "radar",
        "disp_ubicacion":  "ubicacion",
        "seccion_sentido": "sentido",
        "disp_lat":        "lat",
        "disp_lng":        "lng",
        "h_cant_veh":      "volumen_vehicular",
    }, inplace=True)

    # Filas completamente nulas (2024 tiene 6.706)
    df.dropna(how="all", inplace=True)
    df.drop_duplicates(inplace=True)

    # Tipos
    df["fecha"]             = _normalizar_fecha_ausa(df["fecha"])
    df["hora"]              = pd.to_numeric(df["hora"], errors="coerce").astype("Int64")
    df["volumen_vehicular"] = pd.to_numeric(df["volumen_vehicular"], errors="coerce")
    df["lat"]               = _fix_coords(df["lat"])
    df["lng"]               = _fix_coords(df["lng"])
    df["año"]               = año

    # Nulos en sentido → DESCONOCIDO
    df["sentido"] = df["sentido"].fillna("DESCONOCIDO").str.strip().str.upper()

    print(f"  ✅ {año}: {df.shape[0]:,} filas | dups eliminados | nulos: {df.isnull().sum().sum()}")
    return df


def limpiar_ausa_2021(df):
    """2021 tiene estructura completamente distinta y H_Cant_Veh 100% nula → se descarta."""
    # H_Cant_Veh está 100% vacía → este año no aporta volumen vehicular
    print(f"  ⚠️  2021: H_Cant_Veh 100% nula → año descartado del análisis de flujo AUSA")
    return None


def limpiar_flujo_ausa(dfs_por_año):
    """
    Recibe dict {año: df} tal como se cargan en la ingesta.
    Limpia cada uno por separado y los une.
    """
    limpios = []

    for año, df in dfs_por_año.items():
        if df is None:
            continue
        print(f"\nLimpiando AUSA {año}...")
        if año == 2021:
            resultado = limpiar_ausa_2021(df)
        else:
            resultado = limpiar_ausa_2022_2025(df, año)
        if resultado is not None:
            limpios.append(resultado)

    df_final = pd.concat(limpios, ignore_index=True)
    print(f"\n✅ Flujo AUSA unificado: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")
    nulos = df_final.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")

    return df_final


# ── Llamada ──────────────────────────────────────────────
dfs_ausa = {
    2021: df_ausa_2021,
    2022: df_ausa_2022,
    2023: df_ausa_2023,
    2024: df_ausa_2024,
    2025: df_ausa_2025,
}
df_flujo_ausa = limpiar_flujo_ausa(dfs_ausa)


Limpiando AUSA 2021...
  ⚠️  2021: H_Cant_Veh 100% nula → año descartado del análisis de flujo AUSA

Limpiando AUSA 2022...
  ✅ 2022: 106,446 filas | dups eliminados | nulos: 0

Limpiando AUSA 2023...
  ✅ 2023: 237,911 filas | dups eliminados | nulos: 0

Limpiando AUSA 2024...
  ✅ 2024: 231,205 filas | dups eliminados | nulos: 0

Limpiando AUSA 2025...
  ✅ 2025: 607,062 filas | dups eliminados | nulos: 0

✅ Flujo AUSA unificado: 1,182,624 filas × 10 columnas
  → Sin nulos


### Limpieza df_flujo_anillo

* Sin duplicados

Conversiones tipo de dato:
* Columnas → nombres en minúscula
* HORA: object ("31MAR2020:15:00:00") → se separa en fecha (datetime) y hora (entero 0-23)

Correcciones:
* Columna con BOM "ï»¿CODIGO_LOCACION" → renombrada a "codigo_locacion"

Descartes:
* LATITUD/LONGITUD: 3.402 nulos → se eliminan las filas (1,8% del total)

In [7]:
def limpiar_flujo_anillo(df):
    df = df.copy()

    # Renombrar columna con BOM
    df.rename(columns={"ï»¿CODIGO_LOCACION": "codigo_locacion"}, inplace=True)

    # Normalizar nombres de columnas
    df.columns = df.columns.str.lower()

    # Unificar SD/sd → DESCONOCIDO en columnas object
    cols_sd = df.select_dtypes("object").columns
    df[cols_sd] = df[cols_sd].apply(lambda col: col.str.strip().str.upper().replace("SD", "DESCONOCIDO"))

    # HORA: "31MAR2020:15:00:00" → fecha y hora separadas
    hora_dt = pd.to_datetime(df["hora"], format="%d%b%Y:%H:%M:%S", errors="coerce")
    df["fecha"] = hora_dt.dt.normalize()
    df["hora"]  = hora_dt.dt.hour

    # Eliminar filas sin coordenadas (3.402 nulos)
    df.dropna(subset=["latitud", "longitud"], inplace=True)

    print(f"✅ Flujo Anillo limpio: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    nulos_restantes = df.isnull().sum()
    print(nulos_restantes[nulos_restantes > 0].to_string() if nulos_restantes.sum() > 0 else "  → Sin nulos")
    return df

df_flujo_anillo = limpiar_flujo_anillo(df_flujo_anillo)

✅ Flujo Anillo limpio: 186,412 filas × 7 columnas
  → Sin nulos


### Limpieza df_lluvia

* Sin duplicados

Conversiones tipo de dato:
* mm: object → float (milímetros de lluvia acumulados)
* Días: object → float (días con precipitaciones en el mes)
* año: object → entero

Correcciones:
* Columnas renombradas: mm → lluvia_mm, Días → lluvia_dias

Filtro temporal:
* Se conservan solo los años 2021–2025 para alinear con el resto del proyecto

In [8]:
def limpiar_lluvia(df):
    if df is None: return None
    df = df.copy()

    # Renombrar columna con encoding roto si quedó así
    df.columns = [c.encode('latin-1').decode('utf-8') if 'a' in c else c for c in df.columns]
    df.rename(columns={"aÃ±o": "año"}, inplace=True)  # por si acaso

    # Tipos
    df["año"]         = pd.to_numeric(df["año"], errors="coerce").astype("Int64")
    df.rename(columns={"mm": "lluvia_mm", "Días": "lluvia_dias"}, inplace=True)
    df["lluvia_mm"]   = pd.to_numeric(df["lluvia_mm"],   errors="coerce")
    df["lluvia_dias"] = pd.to_numeric(df["lluvia_dias"], errors="coerce")

    # Filtrar solo años relevantes del proyecto
    df = df[df["año"].between(2021, 2025)].reset_index(drop=True)

    print(f"✅ Lluvia limpia: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")
    return df

df_lluvia = limpiar_lluvia(df_lluvia)

✅ Lluvia limpia: 60 filas × 4 columnas
  → Sin nulos


### Limpieza df_temp

* Sin duplicados

Conversiones tipo de dato:
* temp_media: object → float (temperatura media mensual en °C)
* temp_max_media: object → float (media de las máximas en °C)
* temp_min_media: object → float (media de las mínimas en °C)
* año: object → entero

Filtro temporal:
* Se conservan solo los años 2021–2025 para alinear con el resto del proyecto

In [9]:
def limpiar_temp(df):
    if df is None: return None
    df = df.copy()

    # Renombrar columna con encoding roto si quedó así
    df.rename(columns={"aÃ±o": "año"}, inplace=True)

    # Tipos
    df["año"]            = pd.to_numeric(df["año"], errors="coerce").astype("Int64")
    df["temp_media"]     = pd.to_numeric(df["temp_media"],     errors="coerce")
    df["temp_max_media"] = pd.to_numeric(df["temp_max_media"], errors="coerce")
    df["temp_min_media"] = pd.to_numeric(df["temp_min_media"], errors="coerce")

    # Filtrar solo años relevantes del proyecto
    df = df[df["año"].between(2021, 2025)].reset_index(drop=True)

    print(f"✅ Temperatura limpia: {df.shape[0]:,} filas × {df.shape[1]} columnas")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0].to_string() if nulos.sum() > 0 else "  → Sin nulos")
    return df

df_temp = limpiar_temp(df_temp)

✅ Temperatura limpia: 60 filas × 5 columnas
  → Sin nulos


## 5. Integración de datasets (merge por fecha)

In [10]:
def merge_datasets(df_sin, df_lluvia, df_temp, df_flujo_ausa, df_flujo_anillo):
    if df_sin is None:
        print("⚠️  Sin dataset principal de siniestros. Abortando merge.")
        return None

    df = df_sin.copy()

    # ── Merge con precipitaciones (por año + mes) ─────────
    if df_lluvia is not None and 'año' in df_lluvia.columns and 'mes' in df_lluvia.columns:
        df = df.merge(df_lluvia, on=['año', 'mes'], how='left', suffixes=('', '_lluvia'))
        print("✅ Merge con precipitaciones OK")

    # ── Merge con temperatura (por año + mes) ─────────────
    if df_temp is not None and 'año' in df_temp.columns and 'mes' in df_temp.columns:
        df = df.merge(df_temp, on=['año', 'mes'], how='left', suffixes=('', '_temp'))
        print("✅ Merge con temperatura OK")

    # ── Flujo AUSA: agregar volumen promedio por mes ───────
    if df_flujo_ausa is not None and 'volumen_vehicular' in df_flujo_ausa.columns:
        cols_agg = [c for c in ['año', 'mes'] if c in df_flujo_ausa.columns]
        if cols_agg:
            flujo_mensual = (df_flujo_ausa
                             .groupby(cols_agg)['volumen_vehicular']
                             .mean()
                             .reset_index()
                             .rename(columns={'volumen_vehicular': 'flujo_ausa_promedio'}))
            df = df.merge(flujo_mensual, on=cols_agg, how='left')
            print("✅ Merge con flujo AUSA OK")

    # ── Flujo Anillo: agregar volumen promedio por mes ─────
    if df_flujo_anillo is not None and 'volumen_vehicular' in df_flujo_anillo.columns:
        cols_agg = [c for c in ['año', 'mes'] if c in df_flujo_anillo.columns]
        if cols_agg:
            flujo_anillo_mensual = (df_flujo_anillo
                                    .groupby(cols_agg)['volumen_vehicular']
                                    .mean()
                                    .reset_index()
                                    .rename(columns={'volumen_vehicular': 'flujo_anillo_promedio'}))
            df = df.merge(flujo_anillo_mensual, on=cols_agg, how='left')
            print("✅ Merge con flujo Anillo Digital OK")

    print(f"\n📊 Dataset integrado final: {df.shape}")
    return df

df_final = merge_datasets(df_siniestros, df_lluvia, df_temp, df_flujo_ausa, df_flujo_anillo)


NameError: name 'df_siniestros' is not defined

In [ ]:
if df_final is not None:
    print("Columnas del dataset integrado:")
    for col in df_final.columns:
        print(f"  {col}: {df_final[col].dtype}")


## 6. Feature Engineering

In [ ]:
def feature_engineering(df):
    if df is None:
        return None
    df = df.copy()

    # ── Franja horaria ────────────────────────────────────
    if 'hora_num' in df.columns:
        def franja(h):
            if pd.isna(h): return 'DESCONOCIDA'
            h = int(h)
            if 0  <= h < 6:  return 'MADRUGADA'
            if 6  <= h < 12: return 'MAÑANA'
            if 12 <= h < 18: return 'TARDE'
            return 'NOCHE'
        df['franja_horaria'] = df['hora_num'].apply(franja)

    # ── Estación del año ──────────────────────────────────
    if 'mes' in df.columns:
        def estacion(m):
            if pd.isna(m): return 'DESCONOCIDA'
            m = int(m)
            if m in [12, 1, 2]: return 'VERANO'
            if m in [3, 4, 5]:  return 'OTOÑO'
            if m in [6, 7, 8]:  return 'INVIERNO'
            return 'PRIMAVERA'
        df['estacion'] = df['mes'].apply(estacion)

    # ── Indicador de lluvia ───────────────────────────────
    for col_mm in ['mm', 'precipitacion_mm', 'milimetros']:
        if col_mm in df.columns:
            df['hubo_lluvia'] = (df[col_mm] > 0).astype(int)
            break

    # ── Log del flujo vehicular (reduce skewness) ─────────
    for col_f in ['flujo_ausa_promedio', 'flujo_anillo_promedio']:
        if col_f in df.columns:
            df[f'{col_f}_log'] = np.log1p(df[col_f])

    print(f"✅ Feature engineering completado. Shape: {df.shape}")
    return df

df_final = feature_engineering(df_final)


## 7. Análisis exploratorio visual

In [ ]:
if df_final is not None:
    # ── Distribución de gravedad ──────────────────────────
    if 'gravedad' in df_final.columns:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        df_final['gravedad'].value_counts().plot(
            kind='bar', ax=axes[0], color=['#e74c3c', '#e67e22', '#3498db'], edgecolor='black')
        axes[0].set_title('Distribución por Gravedad del Siniestro')
        axes[0].set_xlabel('Gravedad')
        axes[0].set_ylabel('Cantidad')
        axes[0].tick_params(axis='x', rotation=0)

        df_final['gravedad'].value_counts().plot(
            kind='pie', ax=axes[1], autopct='%1.1f%%',
            colors=['#e74c3c', '#e67e22', '#3498db'])
        axes[1].set_title('Proporción por Gravedad')
        axes[1].set_ylabel('')

        plt.tight_layout()
        plt.savefig('outputs/gravedad_distribucion.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Siniestros por franja horaria ─────────────────────
    if 'franja_horaria' in df_final.columns:
        orden = ['MADRUGADA', 'MAÑANA', 'TARDE', 'NOCHE', 'DESCONOCIDA']
        conteo = df_final['franja_horaria'].value_counts().reindex(orden).dropna()

        plt.figure()
        conteo.plot(kind='bar', color='#2c3e50', edgecolor='white')
        plt.title('Siniestros por Franja Horaria')
        plt.xlabel('Franja')
        plt.ylabel('Cantidad')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig('outputs/siniestros_franja_horaria.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Siniestros por mes ────────────────────────────────
    if 'mes' in df_final.columns:
        meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
        conteo_mes = df_final.groupby('mes').size()

        plt.figure()
        conteo_mes.plot(kind='bar', color='#16a085', edgecolor='white')
        plt.title('Siniestros por Mes')
        plt.xlabel('Mes')
        plt.ylabel('Cantidad')
        plt.xticks(ticks=range(12), labels=meses, rotation=0)
        plt.tight_layout()
        plt.savefig('outputs/siniestros_mes.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if df_final is not None:
    # ── Heatmap: día de semana × franja horaria ───────────
    if 'dia_semana' in df_final.columns and 'franja_horaria' in df_final.columns:
        dias = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']
        pivot = df_final.groupby(['dia_semana', 'franja_horaria']).size().unstack(fill_value=0)
        pivot.index = [dias[i] for i in pivot.index if i < 7]

        plt.figure(figsize=(10, 5))
        sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
        plt.title('Siniestros: Día de semana × Franja horaria')
        plt.tight_layout()
        plt.savefig('outputs/heatmap_dia_franja.png', dpi=150, bbox_inches='tight')
        plt.show()


## 8. Mapa de calor geoespacial

In [ ]:
import importlib.util

if df_final is not None and 'lat' in df_final.columns and 'lon' in df_final.columns:
    if importlib.util.find_spec('folium') is not None:
        import folium
        from folium.plugins import HeatMap

        coords = df_final[['lat', 'lon']].dropna()
        # Filtrar coordenadas dentro de CABA (bounding box aproximado)
        coords = coords[
            (coords['lat'].between(-34.75, -34.52)) &
            (coords['lon'].between(-58.55, -58.33))
        ]

        m = folium.Map(location=[-34.61, -58.44], zoom_start=12, tiles='CartoDB positron')
        HeatMap(coords.values.tolist(), radius=10, blur=15, max_zoom=13).add_to(m)

        m.save('outputs/mapa_calor_siniestros.html')
        print("✅ Mapa guardado en outputs/mapa_calor_siniestros.html")
        print(f"   Coordenadas válidas usadas: {len(coords):,}")
        m
    else:
        print("⚠️  Folium no instalado. Ejecutá: pip install folium")
else:
    print("⚠️  Sin columnas lat/lon disponibles para el mapa")


## 9. Modelo predictivo — Pipeline de clasificación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_auc_score)

import os
os.makedirs('outputs', exist_ok=True)


In [ ]:
def preparar_features(df):
    """Selecciona y prepara features y target para el modelo."""
    if df is None:
        return None, None, None, None

    TARGET = 'gravedad'
    if TARGET not in df.columns:
        print(f"⚠️  Columna '{TARGET}' no encontrada. Columnas disponibles: {list(df.columns)}")
        return None, None, None, None

    # Features numéricas potenciales
    FEAT_NUM = [c for c in [
        'hora_num', 'hora_sin', 'hora_cos', 'dia_semana', 'mes',
        'es_fin_semana', 'flujo_ausa_promedio', 'flujo_anillo_promedio',
        'flujo_ausa_promedio_log', 'flujo_anillo_promedio_log',
        'hubo_lluvia'
    ] if c in df.columns]

    # Features categóricas potenciales
    FEAT_CAT = [c for c in [
        'franja_horaria', 'estacion', 'tipo_de_calle',
        'victima', 'acusado', 'sexo'
    ] if c in df.columns]

    print(f"Features numéricas ({len(FEAT_NUM)}): {FEAT_NUM}")
    print(f"Features categóricas ({len(FEAT_CAT)}): {FEAT_CAT}")

    df_model = df[FEAT_NUM + FEAT_CAT + [TARGET]].dropna(subset=[TARGET])

    # Codificar target
    le = LabelEncoder()
    y = le.fit_transform(df_model[TARGET].astype(str))
    X = df_model[FEAT_NUM + FEAT_CAT]

    print(f"\nClases del target: {dict(zip(le.classes_, range(len(le.classes_))))}")
    print(f"Distribución: {pd.Series(y).value_counts().to_dict()}")
    print(f"Shape X: {X.shape}")

    return X, y, le, FEAT_NUM, FEAT_CAT

resultado = preparar_features(df_final)

if resultado[0] is not None:
    X, y, le, FEAT_NUM, FEAT_CAT = resultado


In [ ]:
if 'X' in dir() and X is not None:

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    # ── Preprocessors por tipo de columna ─────────────────
    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # Solo incluir transformadores para columnas que existen
    transformers = []
    if FEAT_NUM:
        transformers.append(('num', num_transformer, FEAT_NUM))
    if FEAT_CAT:
        transformers.append(('cat', cat_transformer, FEAT_CAT))

    preprocessor = ColumnTransformer(transformers=transformers)

    # ── Pipeline completo ─────────────────────────────────
    pipeline_rf = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=100, max_depth=10,
            class_weight='balanced', random_state=42, n_jobs=-1))
    ])

    pipeline_lr = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42))
    ])

    print("✅ Pipelines definidos")
    print("   → pipeline_rf: Random Forest")
    print("   → pipeline_lr: Regresión Logística (baseline)")


In [ ]:
if 'pipeline_rf' in dir():
    print("🔄 Entrenando modelos...")

    # Baseline: Regresión Logística
    pipeline_lr.fit(X_train, y_train)
    y_pred_lr = pipeline_lr.predict(X_test)
    print("\n── REGRESIÓN LOGÍSTICA (Baseline) ───────────────")
    print(classification_report(y_test, y_pred_lr, target_names=le.classes_))

    # Modelo principal: Random Forest
    pipeline_rf.fit(X_train, y_train)
    y_pred_rf = pipeline_rf.predict(X_test)
    print("\n── RANDOM FOREST ────────────────────────────────")
    print(classification_report(y_test, y_pred_rf, target_names=le.classes_))


## 10. Evaluación del modelo

In [ ]:
if 'y_pred_rf' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matriz de confusión — Random Forest
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred_rf),
        display_labels=le.classes_
    ).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title('Matriz de Confusión — Random Forest')

    # Matriz de confusión — Logística
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred_lr),
        display_labels=le.classes_
    ).plot(ax=axes[1], colorbar=False, cmap='Greens')
    axes[1].set_title('Matriz de Confusión — Logística (Baseline)')

    plt.tight_layout()
    plt.savefig('outputs/matrices_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if 'pipeline_rf' in dir():
    # Feature importance del Random Forest
    rf_model = pipeline_rf.named_steps['classifier']
    prep = pipeline_rf.named_steps['preprocessor']

    # Obtener nombres de features post-transformación
    feature_names = []
    for name, transformer, cols in prep.transformers_:
        if name == 'num':
            feature_names.extend(cols)
        elif name == 'cat':
            ohe = transformer.named_steps['ohe']
            feature_names.extend(ohe.get_feature_names_out(cols))

    importances = pd.Series(rf_model.feature_importances_, index=feature_names)
    top15 = importances.nlargest(15)

    plt.figure(figsize=(10, 6))
    top15.sort_values().plot(kind='barh', color='#2980b9', edgecolor='white')
    plt.title('Top 15 Features más Importantes — Random Forest')
    plt.xlabel('Importancia')
    plt.tight_layout()
    plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()


## 11. Guardar resultados

In [ ]:
import pickle

# Dataset final limpio e integrado
if df_final is not None:
    df_final.to_csv('outputs/dataset_final_limpio.csv', index=False)
    print("✅ Dataset guardado en outputs/dataset_final_limpio.csv")

# Modelos entrenados
if 'pipeline_rf' in dir():
    with open('outputs/modelo_random_forest.pkl', 'wb') as f:
        pickle.dump(pipeline_rf, f)
    with open('outputs/modelo_logistica.pkl', 'wb') as f:
        pickle.dump(pipeline_lr, f)
    with open('outputs/label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    print("✅ Modelos guardados en outputs/")

print("\n🎉 Pipeline completo ejecutado exitosamente")


## 12. Resumen del Pipeline

```
[1] INGESTA          → Carga de 5 CSVs oficiales (GCBA)
       ↓
[2] EXPLORACIÓN      → Shape, tipos, nulos, duplicados
       ↓
[3] LIMPIEZA         → Normalización de columnas, fechas, coordenadas,
                       valores SD → NaN, texto a mayúsculas
       ↓
[4] INTEGRACIÓN      → Merge por año + mes con clima y flujo vehicular
       ↓
[5] FEATURE ENG.     → Franja horaria, estación, hubo_lluvia,
                       variables cíclicas hora (sin/cos), log flujo
       ↓
[6] EDA VISUAL       → Distribuciones, heatmap, mapa folium
       ↓
[7] MODELO           → ColumnTransformer + Pipeline sklearn
                       Baseline: Regresión Logística
                       Principal: Random Forest Classifier
       ↓
[8] EVALUACIÓN       → Classification report, matrices de confusión,
                       feature importance
       ↓
[9] PERSISTENCIA     → CSV limpio + modelos .pkl
```

### 📁 Archivos generados en `outputs/`
| Archivo | Descripción |
|---|---|
| `dataset_final_limpio.csv` | Dataset integrado y limpio |
| `mapa_calor_siniestros.html` | Mapa interactivo de accidentes |
| `gravedad_distribucion.png` | Distribución del target |
| `siniestros_franja_horaria.png` | Siniestros por hora del día |
| `siniestros_mes.png` | Siniestros por mes |
| `heatmap_dia_franja.png` | Heatmap día × franja |
| `matrices_confusion.png` | Evaluación de modelos |
| `feature_importance.png` | Variables más predictivas |
| `modelo_random_forest.pkl` | Modelo RF entrenado |
| `modelo_logistica.pkl` | Modelo LR entrenado |
| `label_encoder.pkl` | Encoder del target |
